Import libraries

In [64]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [65]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 25 ms, sys: 8.01 ms, total: 33 ms
Wall time: 32.3 ms
CPU times: user 11.6 ms, sys: 4 ms, total: 15.6 ms
Wall time: 15.3 ms
CPU times: user 4.24 ms, sys: 1 ms, total: 5.24 ms
Wall time: 5.25 ms


In [66]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [67]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [68]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [69]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [70]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [71]:
epochs = 150
lr = 5e-5
wd = 1e-5
alpha = 0.1
beta = 0.1
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
shared_dropout = 0
outcome_droupout = 0.0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 learning rate = 5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30


In [72]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        alpha=alpha, beta=beta,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_droupout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    dragonnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini


Epoch 1/150 | Base Loss: 415.0600 | Tarreg Loss: 20.996571 | Total Loss: 436.0566 | Val Loss: 499.0307 | Raw Qini: 0.4056 | EMA Trend: 0.4056 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 537.7202 | Tarreg Loss: 27.097300 | Total Loss: 564.8175 | Val Loss: 499.0017 | Raw Qini: 0.4388 | EMA Trend: 0.4106 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 288.3089 | Tarreg Loss: 14.511472 | Total Loss: 302.8203 | Val Loss: 498.9729 | Raw Qini: 0.4786 | EMA Trend: 0.4208 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 262.3111 | Tarreg Loss: 13.280306 | Total Loss: 275.5914 | Val Loss: 498.9442 | Raw Qini: 0.5075 | EMA Trend: 0.4338 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 656.3097 | Tarreg Loss: 33.259083 | Total Loss: 689.5688 | Val Loss: 498.9146 | Raw Qini: 0.5575 | EMA Trend: 0.4524 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 304.7934 | Tarreg Loss: 15.459078 | Total Loss: 320.2525 | Val Loss: 498.8834 | Raw Qini: 0.5786 | EMA Trend: 0.4713 | ⭐ NEW 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 423.0125 | Tarreg Loss: 21.303246 | Total Loss: 444.3158 | Val Loss: 498.5630 | Raw Qini: 0.6110 | EMA Trend: 0.6110 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 497.3170 | Tarreg Loss: 24.806040 | Total Loss: 522.1231 | Val Loss: 498.5342 | Raw Qini: 0.5815 | EMA Trend: 0.6066 | (patience: 1/20)
Epoch 3/150 | Base Loss: 515.6705 | Tarreg Loss: 25.778666 | Total Loss: 541.4492 | Val Loss: 498.5051 | Raw Qini: 0.5101 | EMA Trend: 0.5921 | (patience: 2/20)
Epoch 4/150 | Base Loss: 348.5974 | Tarreg Loss: 17.541096 | Total Loss: 366.1385 | Val Loss: 498.4747 | Raw Qini: 0.4546 | EMA Trend: 0.5715 | (patience: 3/20)
Epoch 5/150 | Base Loss: 357.1292 | Tarreg Loss: 17.912334 | Total Loss: 375.0416 | Val Loss: 498.4431 | Raw Qini: 0.3933 | EMA Trend: 0.5448 | (patience: 4/20)
Epoch 6/150 | Base Loss: 432.9145 | Tarreg Loss: 21.725828 | Total Loss: 454.6403 | Val Loss: 498.4111 | Raw Qini: 0.2945 | EMA Trend: 0.5072 | (patience: 5/20)
Epoch 7/150 | Base Loss: 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 408.2905 | Tarreg Loss: 20.890171 | Total Loss: 429.1807 | Val Loss: 498.6142 | Raw Qini: 0.2693 | EMA Trend: 0.2693 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 353.0642 | Tarreg Loss: 18.261244 | Total Loss: 371.3254 | Val Loss: 498.5833 | Raw Qini: 0.2757 | EMA Trend: 0.2703 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 372.7642 | Tarreg Loss: 19.251495 | Total Loss: 392.0157 | Val Loss: 498.5522 | Raw Qini: 0.2722 | EMA Trend: 0.2706 | ✓ above trend but not peak (patience: 1/20)
Epoch 4/150 | Base Loss: 441.7018 | Tarreg Loss: 22.507774 | Total Loss: 464.2096 | Val Loss: 498.5199 | Raw Qini: 0.2968 | EMA Trend: 0.2745 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 482.7222 | Tarreg Loss: 24.716688 | Total Loss: 507.4389 | Val Loss: 498.4860 | Raw Qini: 0.3443 | EMA Trend: 0.2850 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 551.9033 | Tarreg Loss: 27.982162 | Total Loss: 579.8854 | Val Loss: 498.4502 | Raw Qini: 0.4007 | EMA Tren

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 439.7898 | Tarreg Loss: 22.203602 | Total Loss: 461.9934 | Val Loss: 499.1789 | Raw Qini: 0.6244 | EMA Trend: 0.6244 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 336.0956 | Tarreg Loss: 17.101889 | Total Loss: 353.1975 | Val Loss: 499.1446 | Raw Qini: 0.6283 | EMA Trend: 0.6250 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 713.2385 | Tarreg Loss: 35.661060 | Total Loss: 748.8996 | Val Loss: 499.1112 | Raw Qini: 0.6216 | EMA Trend: 0.6245 | (patience: 1/20)
Epoch 4/150 | Base Loss: 553.0241 | Tarreg Loss: 27.896513 | Total Loss: 580.9206 | Val Loss: 499.0782 | Raw Qini: 0.6445 | EMA Trend: 0.6275 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 437.3498 | Tarreg Loss: 21.941282 | Total Loss: 459.2911 | Val Loss: 499.0446 | Raw Qini: 0.6677 | EMA Trend: 0.6335 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 374.8144 | Tarreg Loss: 19.007338 | Total Loss: 393.8217 | Val Loss: 499.0099 | Raw Qini: 0.6827 | EMA Trend: 0.6409 | ⭐ NEW BEST (pea

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
